# Twelve recommenders. Now build *the* recommender.

You have built twelve kinds of recommender: popularity (Unit 2), BM25 (U3), item-item CF (U4), matrix
factorization (U5), a calibrated blend (U6), semantic embeddings (U7), the two-tower (U8), the feature
tower for cold books (U9), an HNSW index and a hybrid (U10), a learned reranker (U11) and a
self-attention sequence model (U12). Unit 13 showed that every one of them is also an **exposure**
decision.

A real team does not ship twelve recommenders. It ships **one system**, end to end:

```
  8 retrieval paths  --->  candidate pool  --->  blend / learned reranker  --->  top 10
  (each proposes 50)       (union, seen books       (orders the pool)
                            removed)
```

and then it has to answer the question this unit is about: **what does each stage actually buy, and at
what cost?** A stage that does not earn its place with a measurement is just cost -- training time,
serving time, things that can break.

So this unit **assembles the whole Part-2 stack** on the main log and measures it the way Unit 13 taught
us to: hit@10 *and* coverage, head-share and the exposure Gini. Two promises, as always:

- **val only.** Every number is on the val split. The sealed final split was opened once at each
  checkpoint, and it stays sealed.
- **One cohort, one rule.** The same 500 warm val readers as Units 6-13, top 10, train books excluded.
  The one exception is the sequence path, which lives on the **session log** (design section 6), so its
  numbers are never put in the same column as the others.

By the end you will have a full-system scoreboard, an ablation table, a beyond-accuracy table and a
**frontier** -- and you will see that "more machinery" is not the same as "better". That frontier is what
you defend in the capstone.

## 0. Setup: the data and the model cache

Several of our models take tens of seconds to train (the reranker over eight paths takes over a minute).
Retraining all of them every time we open a notebook would be slow, and it is not what a production
team does either: models are **trained once by a pipeline and loaded by everything downstream**.

Our pipeline is `recsys/projects/bookrec/scripts/build_models.py`. It fits every expensive model from the
same seeded code you used in Units 5-12 and saves each fitted model's `artifact()` (the plain numpy state
you met in Unit 8) into a **model cache**. `load_cached_models()` reads that cache back. Two safety rules
are built in:

- the cache records a fingerprint of the generated data, and it **refuses to load** if the data has
  changed since it was built (a stale model on new data is a silent bug);
- if the cache is missing, the error message names the build command.

The cache is rebuilt from scratch on every CI run, and a required check compares it against numbers
measured independently. Below we go one step further and **fit two of the models on screen** to see that
the cached ones are the same models.

`load_book_data()` loads the catalog, keywords, GloVe subset and main log, and the val cohort -- the
same 500 readers as Unit 13. `top10` and `hit_at_10` are the scoreboard's rule, written out once more.

In [1]:
import numpy as np
from bookrec import (
    AnnRetrievalPath,
    FixedBlendPath,
    FullSystem,
    MatrixFactorizationPath,
    NeuralRerankerPath,
    RerankerModel,
    TwoTowerRetrievalPath,
    blend,
    catalog_coverage,
    load_book_data,
    load_cached_models,
    rank,
)
from bookrec.rerank import family_mask, feature_names

K = 10
POOL = 50
SEED = 0

data = load_book_data()
catalog, catalog_ids, rows = data.catalog, data.catalog_ids, data.rows
cohort, seen_by, relevant = data.cohort, data.seen_by, data.relevant
cache = load_cached_models()


def top10(path, reader):
    # The scoreboard's rule for one reader: retrieve, drop train books, keep 10.
    seen = seen_by.get(reader, set())
    return [c.item_id for c in rank(path.retrieve(reader, {"seen": seen}, K), n=K, exclude=seen)]


def hit_at_10(lists):
    # Mean per-reader hit: 1.0 if any unseen val book is in the reader's top 10.
    return float(np.mean([float(bool(set(lists[r]) & relevant[r])) for r in lists]))


print(f"catalog {len(catalog_ids)} books; {len(rows)} main-log rows; val cohort {len(cohort)} readers")
print(f"model cache: {len(cache.envelopes)} cached models, built from THIS data (fingerprint checked)")
for name in cache.envelopes:
    envelope = cache.envelope(name)
    print(f"  {name:<19} {envelope['path_class']}")

catalog 2000 books; 27004 main-log rows; val cohort 500 readers
model cache: 8 cached models, built from THIS data (fingerprint checked)
  mf                  MatrixFactorizationPath
  two-tower           TwoTowerRetrievalPath
  feature-tower       FeatureTowerRetrievalPath
  feature-tower-hard  FeatureTowerRetrievalPath
  ann                 AnnRetrievalPath
  reranker-u11        NeuralRerankerPath
  reranker-full8      NeuralRerankerPath
  sasrec              SasrecRetrievalPath


## 1. Retrieval, part 1: fit two models on screen, then trust the cache

### Matrix factorization and the two-tower, fitted live

These are the two collaborative models every later stage leans on: MF (Unit 5, hit about 0.276) and the
two-tower (Unit 8, hit about 0.340). We fit each once, from the same seeded recipe the build script uses,
and compare it with the cached copy two ways:

- **the numbers inside** -- every learned array, `np.allclose` to the cached one;
- **what a reader sees** -- the top 10 of the first 20 cohort readers, identical.

(Each fit takes roughly 15-30 seconds; these two cells are the unit's only long ones.)

In [2]:
mf = MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)
mf_cached = MatrixFactorizationPath(seed=SEED).load(cache.artifact("mf"))

live, stored = mf.artifact(), cache.artifact("mf")
same_arrays = all(np.allclose(live[key], stored[key]) for key in ("P", "Q"))
first20 = cohort[:20]
same_lists = sum(top10(mf, r) == top10(mf_cached, r) for r in first20)
print(f"MF fitted on screen: P {live['P'].shape}, Q {live['Q'].shape}")
print(f"allclose to the cached MF on every learned array: {same_arrays}")
print(f"identical top 10 for {same_lists} of {len(first20)} readers")

MF fitted on screen: P (540, 32), Q (2000, 32)
allclose to the cached MF on every learned array: True
identical top 10 for 20 of 20 readers


In [3]:
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)
two_tower_cached = TwoTowerRetrievalPath(seed=SEED).load(cache.artifact("two-tower"))

live, stored = two_tower.artifact(), cache.artifact("two-tower")
same_arrays = all(
    np.allclose(live[key], stored[key]) for key in ("reader_embeddings", "item_embeddings")
)
same_lists = sum(top10(two_tower, r) == top10(two_tower_cached, r) for r in first20)
loss = two_tower.loss_history
print(f"two-tower fitted on screen: {len(loss)} epochs, loss {loss[0]:.3f} -> {loss[-1]:.3f}")
print(f"allclose to the cached two-tower on both embedding tables: {same_arrays}")
print(f"identical top 10 for {same_lists} of {len(first20)} readers")
print(f"the cache even stores the training curve: {np.allclose(loss, cache.loss_history('two-tower'))}")

two-tower fitted on screen: 60 epochs, loss 0.370 -> 0.111
allclose to the cached two-tower on both embedding tables: True
identical top 10 for 20 of 20 readers
the cache even stores the training curve: True


Same arrays, same lists. From here on, every expensive model is **loaded**, not refit: the feature
tower (U9), both learned rerankers (U11) and SASRec (U12). You already watched those train in their own
units; here we read their stored training curves instead.

### Serving the two-tower through HNSW

In a real catalog of millions, scoring every book for every reader is too slow, so the full stack
serves the two-tower through Unit 10's **HNSW** graph index. Building it takes well under a second. The
question to ask is Unit 10's: **does the approximate index return what the exact scan returns?**

We measure that the honest way, by hand: for each cohort reader, the exact top 10 (scan all 2,000 books)
against the ANN top 10, and their overlap. That mean overlap is **recall@10 vs exact**.

In [4]:
ann = AnnRetrievalPath(two_tower, efSearch=64, m=32, ef_construction=200).fit(rows, catalog=catalog_ids)

recall, exact_lists, ann_lists = [], {}, {}
for r in cohort:
    exact_lists[r], ann_lists[r] = top10(two_tower, r), top10(ann, r)
    recall.append(len(set(exact_lists[r]) & set(ann_lists[r])) / K)
recall_at_10 = float(np.mean(recall))
print(f"HNSW over the two-tower: {ann.index.size} book vectors indexed")
print(f"recall@10 vs the exact scan, over {len(cohort)} readers: {recall_at_10:.4f}")
print(f"readers whose ANN top 10 differs from exact at all: {sum(x < 1 for x in recall)}")
print(f"hit@10  exact two-tower {hit_at_10(exact_lists):.3f}   ANN {hit_at_10(ann_lists):.3f}")

ann_cached = AnnRetrievalPath(two_tower_cached).load(cache.artifact("ann"))
same = all(top10(ann_cached, r) == ann_lists[r] for r in cohort)
print(f"the cached index (rebuilt on load) serves identical top 10s for all {len(cohort)}: {same}")

HNSW over the two-tower: 2000 book vectors indexed
recall@10 vs the exact scan, over 500 readers: 0.9988
readers whose ANN top 10 differs from exact at all: 6
hit@10  exact two-tower 0.340   ANN 0.340
the cached index (rebuilt on load) serves identical top 10s for all 500: True


Recall about **0.999**: on a handful of readers one book in ten differs, and the hit is the same 0.340.
That is the Unit-10 lesson, now inside a full system: **ANN is a speed tool, not an accuracy tool.** It
earns its place only when the catalog is big enough that the exact scan is too slow -- at 2,000 books
it is not, so in the ablation table below "exact instead of ANN" should cost almost nothing. If it ever
*improved* hit, that would be a coincidence of which book got swapped, not a reason to prefer it.

## 2. Assemble the system

`FullSystem.from_cache()` does what we just did for every path at once: it refits the four cheap paths
(popularity, BM25, item-item, semantic -- a second or so), loads MF, the two-tower, both feature towers,
the HNSW index, both rerankers and SASRec from the cache, and builds the U10 hybrid twice (over the exact
two-tower, and over the ANN one). `system.full8` holds the **eight serving paths** of the full stack;
the two-tower and the hybrid in it are served through HNSW.

`system.readout(lists)` reads every number Unit 13 taught (hit, coverage, head-share, item and author
Gini, novelty, intra-list diversity) off one set of top-10 lists. First we check that its hit and
coverage agree with our by-hand helpers on the ANN lists.

In [5]:
system = FullSystem.from_cache(cache=cache, data=data)
print("the 8 serving paths of the full stack:")
for name, path in system.full8.items():
    print(f"  {name:<14} served by: {path.name}")

check = system.readout(ann_lists)
assert abs(check["hit"] - hit_at_10(ann_lists)) < 1e-12
assert abs(check["coverage"] - catalog_coverage(ann_lists.values(), catalog_ids)) < 1e-12
print("\nsystem.readout agrees with the by-hand hit@10 and coverage")

the 8 serving paths of the full stack:
  popularity     served by: popularity
  lexical        served by: lexical
  item-item      served by: item-item
  mf             served by: mf
  semantic       served by: semantic
  two-tower      served by: ann
  feature-tower  served by: feature-tower
  hybrid         served by: hybrid

system.readout agrees with the by-hand hit@10 and coverage


Each path alone, read on the same 500 readers. These are the numbers Units 2-13 reported; the point of
re-reading them here is to have the **single-path bar** in front of us before we combine anything. One
detail: in the stack, the two-tower and the hybrid are the **HNSW-served** versions.

In [6]:
print(f"{'path':<14} {'hit@10':>7} {'coverage':>9} {'head-share':>11}")
for name, path in system.full8.items():
    m = system.evaluate(path)
    print(f"{name:<14} {m['hit']:>7.3f} {m['coverage']:>9.4f} {m['head_share']:>11.3f}")

path            hit@10  coverage  head-share
popularity       0.108    0.0080       1.000
lexical          0.158    0.2780       0.268
item-item        0.252    0.4900       0.491
mf               0.276    0.0745       0.994
semantic         0.102    0.3545       0.109
two-tower        0.340    0.1775       0.900
feature-tower    0.298    0.3065       0.638
hybrid           0.360    0.1915       0.864


The hybrid (BM25 + two-tower) is the best single path. Served over HNSW it scores 0.360 -- exactly one
reader in 500 below the exact-scan hybrid's **0.362** (Unit 10's number), which is the swapped-book
noise we just saw. MF and popularity are the most concentrated; the content paths reach furthest at the
lowest hit. Hold on to two numbers for the **best single path**, the exact hybrid: **0.362** hit at about
**0.19** coverage.

## 3. Retrieval, part 2: the feature tower for cold books

Every collaborative path we have needs **reads**: a book with no train reads has no learned vector and
can never be recommended (Unit 13 counted 0 for item-item, MF and the two-tower). The **feature tower**
(Unit 9) builds a book's vector from its **features** -- genres, author, keywords -- so it can serve a
book nobody has read yet. That is why it is in the stack.

Unit 9 also showed its knob: which **negatives** it trains against. The shipped tower draws negatives
from warm books (`negative_pool="warm"`). With **hard negatives** (`"hard"`) it is pushed to separate
look-alike books, which spreads its scores into the cold catalog. The cache holds **both** fits; the
served stack uses the warm one. Here is the trade, read off the cached models with Unit 9's two readouts: hit@10 on the 500-reader
cohort, and **cold coverage** -- the fraction of the zero-read books that appear in *any* warm reader's
top 10 (every reader the tower was fitted on, not only the 500 val readers, so the number is a little
larger than Unit 13's cohort-only count).

In [7]:
train_items = {i for items in seen_by.values() for i in items}
zero_train = set(catalog_ids) - train_items          # books with no train reads at all
warm_readers = [r for r in cache.artifact("feature-tower")["reader_ids"] if r not in set(data.cold_readers)]
print(f"{len(zero_train)} catalog books have zero train reads; "
      f"Unit 9's cold readout serves all {len(warm_readers)} warm readers the tower knows\n")

print(f"{'feature tower':<26} {'hit@10':>7} {'cold books surfaced':>20} {'cold coverage':>14}")
for label, name in (("warm negatives (served)", "feature-tower"), ("hard negatives", "feature-tower-hard")):
    tower = system.paths[name]
    surfaced = {i for r in warm_readers for i in top10(tower, r)} & zero_train
    print(f"{label:<26} {hit_at_10(system.top_lists(tower)):>7.3f} {len(surfaced):>20} "
          f"{len(surfaced) / len(zero_train):>14.3f}")

for name in ("feature-tower", "feature-tower-hard"):
    curve = cache.loss_history(name)
    print(f"\n{name} training curve (cached): {len(curve)} epochs, "
          f"loss {curve[0]:.3f} -> {curve[-1]:.3f}", end="")
print()

818 catalog books have zero train reads; Unit 9's cold readout serves all 540 warm readers the tower knows

feature tower               hit@10  cold books surfaced  cold coverage
warm negatives (served)      0.298                  113          0.138
hard negatives               0.172                  185          0.226

feature-tower training curve (cached): 40 epochs, loss 0.219 -> 0.069
feature-tower-hard training curve (cached): 40 epochs, loss 0.247 -> 0.071


- With **warm negatives** the tower keeps most of the collaborative accuracy (hit about **0.298**) and
  still surfaces **113** of the 818 zero-read books (cold coverage about **0.138**) -- something no
  other learned path in the stack does.
- With **hard negatives** cold coverage rises to about **0.226**, but warm hit collapses to about
  **0.172**.

That is a **trade**, not an improvement: harder negatives buy reach into the cold catalog with accuracy
on the readers we can score. We serve the warm tower and keep the hard one as an ablation. Whether a
new author's first book deserves that price is exactly the kind of question the capstone memo asks you
to take a position on.

## 4. The candidate pool and the fixed blend

Now combine. Each of the 8 paths proposes its top **50** for a reader; the **pool** is their union;
Unit 6's `blend` min-max calibrates each path's scores to $[0, 1]$ and adds them up (equal weights);
the reader's train books are dropped and the top 10 kept. We do it by hand for one reader and check it
against `system.fixed_blend()`:

In [8]:
reader = cohort[0]
seen = seen_by[reader]
per_path = {name: path.retrieve(reader, {"seen": seen}, POOL) for name, path in system.full8.items()}
pool_ids = {c.item_id for cands in per_path.values() for c in cands}
by_hand = [c.item_id for c in blend(per_path) if c.item_id not in seen][:K]

fixed_blend = system.fixed_blend()
print(f"reader {reader}: 8 paths x {POOL} proposals -> a pool of {len(pool_ids)} distinct books")
print(f"fixed blend top 10 by hand:      {by_hand}")
print(f"system.fixed_blend() top 10:     {top10(fixed_blend, reader)}")
assert by_hand == top10(fixed_blend, reader)

pool_recall = np.mean([
    float(bool({c.item_id for p in system.full8.values()
                for c in p.retrieve(r, {"seen": seen_by[r]}, POOL)} & relevant[r]))
    for r in cohort
])
print(f"\npool recall (a val book anywhere in the pool): {pool_recall:.3f} -- the ceiling for ANY ordering")
fixed = system.evaluate(fixed_blend)
best = system.evaluate(system.best_single_path)       # the U10 hybrid over the exact two-tower
print(f"fixed 8-path blend:              hit@10 {fixed['hit']:.3f}   coverage {fixed['coverage']:.4f}")
print(f"best single path (exact hybrid): hit@10 {best['hit']:.3f}   coverage {best['coverage']:.4f}")

reader 1: 8 paths x 50 proposals -> a pool of 234 distinct books
fixed blend top 10 by hand:      [452, 1743, 505, 1415, 925, 1497, 818, 1050, 184, 1309]
system.fixed_blend() top 10:     [452, 1743, 505, 1415, 925, 1497, 818, 1050, 184, 1309]

pool recall (a val book anywhere in the pool): 0.800 -- the ceiling for ANY ordering
fixed 8-path blend:              hit@10 0.362   coverage 0.2060
best single path (exact hybrid): hit@10 0.362   coverage 0.1920


Two things to read. First, the **pool recall** is far above any top-10 hit: the right book is usually
*somewhere* in the 8 x 50 proposals, and the whole job of the ordering stage is to lift it into the top
10. Second, the equal-weight fixed blend **ties the hybrid's hit (0.362) and reaches a little further
(coverage about 0.206 vs 0.192)**. Eight paths together did not beat the best one on hit -- but they did
not lose to it either, and they spread the shelf a little.

## 5. The learned reranker over the full pool

Unit 11's question, asked again with all eight paths: can a **learned** model order the pool better than
the fixed sum? Each candidate in the pool becomes a vector of **ranking features**:

In [9]:
path_names = tuple(system.full8)
names = feature_names(path_names)
meta = cache.meta("reranker-full8")
print(f"{len(names)} ranking features per candidate:")
print(f"  scores (8):   {names[0:8]}")
print(f"  presence (8): {names[8:16]}")
print(f"  meta (1):     {names[16:17]}")
print(f"  content (4):  {names[17:21]}")
print(f"\ncached reranker-full8: recipe {meta['recipe']!r}, profile fraction {meta['profile_frac']}, "
      f"{meta['training_rows']} training rows, {meta['n_features']} features")

21 ranking features per candidate:
  scores (8):   ['score:popularity', 'score:lexical', 'score:item-item', 'score:mf', 'score:semantic', 'score:two-tower', 'score:feature-tower', 'score:hybrid']
  presence (8): ['in:popularity', 'in:lexical', 'in:item-item', 'in:mf', 'in:semantic', 'in:two-tower', 'in:feature-tower', 'in:hybrid']
  meta (1):     ['n_paths']
  content (4):  ['genre_frac', 'genre_cos', 'author_frac', 'log_pop']

cached reranker-full8: recipe 'u11-holdout-75-25', profile fraction 0.75, 7458 training rows, 21 features


The recipe is **Unit 11's leakage-safe recipe, unchanged**: per reader, the time-ordered latest 25% of
train reads are the **labels**, the earlier 75% are the **profile**; all 8 paths are **refit on the
profile only**, so no path has seen a label book when it scores one; and a small MLP learns to rank
(Unit 11's **learning to rank**: held-out positives against sampled pool negatives). Refitting eight
paths on the profile is what makes this the most expensive model in the book, so it comes from the cache
-- with its **training log**:

In [10]:
log = cache.loss_history("reranker-full8")
print(f"reranker-full8 training log ({len(log)} epochs of the MLP):")
for epoch in (0, 4, 9, 14, 19, 24, 29):
    print(f"  epoch {epoch + 1:>2}  loss {log[epoch]:.4f}")

full = system.evaluate(system.full_stack)
u11 = system.evaluate(system.reranker_u11)
print(f"\n{'system':<38} {'hit@10':>7} {'coverage':>9}")
print(f"{'fixed 8-path blend (no reranker)':<38} {fixed['hit']:>7.3f} {fixed['coverage']:>9.4f}")
print(f"{'FULL STACK: 8-path pool + reranker':<38} {full['hit']:>7.3f} {full['coverage']:>9.4f}")
print(f"{'U11 reranker (6 paths, reference)':<38} {u11['hit']:>7.3f} {u11['coverage']:>9.4f}")
print(f"\nreranker minus fixed blend: hit {full['hit'] - fixed['hit']:+.3f}, "
      f"coverage {full['coverage'] - fixed['coverage']:+.4f}")

reranker-full8 training log (30 epochs of the MLP):
  epoch  1  loss 0.4124
  epoch  5  loss 0.2384
  epoch 10  loss 0.2360
  epoch 15  loss 0.2344
  epoch 20  loss 0.2306
  epoch 25  loss 0.2298
  epoch 30  loss 0.2291

system                                  hit@10  coverage
fixed 8-path blend (no reranker)         0.362    0.2060
FULL STACK: 8-path pool + reranker       0.356    0.2025
U11 reranker (6 paths, reference)        0.346    0.1945

reranker minus fixed blend: hit -0.006, coverage -0.0035


The loss falls smoothly -- the model trained fine. And yet the full stack scores **0.356, below the fixed
blend's 0.362**, with slightly less reach. The learned reranker **dilutes** the signal, exactly as in
Unit 11, where the 6-path reranker (**0.346**, shown for reference) only tied the fixed 6-way order.
Unit 11 found why: the path scores are highly correlated with each other, and the MLP spends its capacity
on them; the one feature family carrying new information (content affinity) gets averaged in with the
rest. Adding two more correlated paths did not change that.

This is the most important result of the unit, so say it plainly: **the most complex stage in the system
does not earn its place on this data.** A real team would measure this and either fix it (Section 9 shows
one direction) or remove the stage. "We have a neural reranker" is not a reason to keep one.

## 6. The sequence path, on its own log

The last piece is Unit 12's **self-attention** model, SASRec. It reads a reader's history **in order**
and predicts the next book, so it needs a log where order means something: the **session log**, with its
own readers and its own val split (design section 6). Its numbers answer a different question on a
different population, so they get their own table -- never a row next to the main-log systems.

In [11]:
session_seen, session_relevant, session_cohort = data.session_cohort
curve = cache.loss_history("sasrec")
print(f"SASRec (cached): {len(curve)} epochs, loss {curve[0]:.2f} -> {curve[-1]:.2f}")

sas_lists = system.session_top_lists()
sas_hit = float(np.mean([float(bool(set(sas_lists[r]) & session_relevant[r])) for r in session_cohort]))
print(f"session log, val: {len(session_cohort)} readers (main log: {len(cohort)} -- a different population)")
print(f"SASRec hit@10 on the session log: {sas_hit:.4f}")

SASRec (cached): 30 epochs, loss 7.30 -> 5.06
session log, val: 1419 readers (main log: 500 -- a different population)
SASRec hit@10 on the session log: 0.4228


SASRec reaches about **0.423** on the session log -- higher than anything on the main log, and it would
be a mistake to call it "the best model in the book". It is scored on 1,419 *different* readers whose
logs were generated with strong reading order (series, sequels). Unit 12 measured its value the right
way: against item-item and an order-shuffled control *on the same log*. In a deployed system it would be
a separate surface ("read next", after finishing a book), not one more vote in the main blend.

## 7. The ablation table: what does each stage buy?

An **ablation** removes one piece and measures what changes. The build script computed the whole table
from the freshly built cache and saved it as `readouts.json`; scoring ten variants costs about a minute,
so we **read** the rows here and check them, rather than recompute them. Two kinds of row, with
**different baselines** -- read the `baseline` column every time:

- **stage rows** remove a *stage* of the full stack (the reranker, or ANN serving) and are compared with
  the **full stack** (0.356);
- **path rows** remove one *path* from the **fixed 8-path blend** -- a fixed-blend **leave-one-out** --
  and are compared with the **fixed blend** (0.362). They are *not* reranked: the reranker was trained
  with all 8 paths' features, so dropping a path under it would measure a broken model, not the path.

First, the saved baselines must be the systems we just measured live:

In [12]:
readouts = cache.readouts
baselines = readouts["baselines"]
assert abs(baselines["full-stack"]["hit"] - full["hit"]) < 1e-12
assert abs(baselines["fixed-blend"]["hit"] - fixed["hit"]) < 1e-12
assert abs(baselines["fixed-blend"]["coverage"] - fixed["coverage"]) < 1e-12
print(f"readouts.json: split {readouts['split']!r}, k {readouts['k']}, {readouts['cohort_size']} readers; "
      "baselines == the live full stack and fixed blend\n")

print(f"{'row':<22} {'baseline':<12} {'hit@10':>7} {'d hit':>7} {'coverage':>9} {'d coverage':>11}")
for name in ("full-stack", "fixed-blend"):
    b = baselines[name]
    print(f"{name:<22} {'(baseline)':<12} {b['hit']:>7.3f} {'':>7} {b['coverage']:>9.4f}")
for row in readouts["ablation"]:
    base = baselines[row["baseline"]]
    assert abs(row["d_hit"] - (row["hit"] - base["hit"])) < 1e-9          # recompute every delta
    assert abs(row["d_coverage"] - (row["coverage"] - base["coverage"])) < 1e-9
    print(f"{row['row']:<22} {row['baseline']:<12} {row['hit']:>7.3f} {row['d_hit']:>+7.3f} "
          f"{row['coverage']:>9.4f} {row['d_coverage']:>+11.4f}")

readouts.json: split 'val', k 10, 500 readers; baselines == the live full stack and fixed blend

row                    baseline      hit@10   d hit  coverage  d coverage
full-stack             (baseline)     0.356            0.2025
fixed-blend            (baseline)     0.362            0.2060
no-reranker            full-stack     0.362  +0.006    0.2060     +0.0035
exact-two-tower        full-stack     0.354  -0.002    0.2025     +0.0000
minus-popularity       fixed-blend    0.370  +0.008    0.2200     +0.0140
minus-lexical          fixed-blend    0.362  +0.000    0.1750     -0.0310
minus-item-item        fixed-blend    0.354  -0.008    0.1920     -0.0140
minus-mf               fixed-blend    0.354  -0.008    0.2375     +0.0315
minus-semantic         fixed-blend    0.358  -0.004    0.1855     -0.0205
minus-two-tower        fixed-blend    0.352  -0.010    0.2210     +0.0150
minus-feature-tower    fixed-blend    0.364  +0.002    0.2075     +0.0015
minus-hybrid           fixed-blend    0

Read it row by row, always against the row's own baseline:

- **no-reranker: +0.006 hit.** Removing the most expensive stage makes the system *better*. That is
  Section 5's result as an ablation row.
- **exact-two-tower: -0.002.** Swapping HNSW for the exact scan changes one reader in 500 -- noise. ANN
  costs nothing in accuracy, as it should.
- **The path rows are all small** (within about 0.01 of hit) and their signs **differ**. Dropping the
  two-tower costs the most hit (-0.010) but *widens* the shelf; dropping lexical costs no hit but the
  most coverage (-0.031) -- BM25 is the blend's reach. Dropping **popularity raises hit by 0.008** and
  coverage too: in an equal-weight blend, the bestseller list mostly crowds out better candidates. We
  report that as measured; we did not tune the stack to make every path look useful.

With eight correlated paths, no single path is indispensable. That is a real finding about
**redundancy**, and a reason a capstone design might ship *fewer* paths.

## 8. Beyond accuracy, and the Part-1 benchmark

The last fixed comparison is against where we started Part 2: the **Part-1 build**, Unit 6's weighted
blend of popularity, BM25, item-item and MF (pool 30), which scored **0.306** at coverage **0.333**. We
recompute it live -- it is the benchmark everything in Part 2 has to be measured against -- and read the
beyond-accuracy table (Unit 13's columns) from `readouts.json`.

In [13]:
part1 = system.evaluate(system.part1_build())
saved = {row["system"]: row for row in readouts["beyond_accuracy"]}
assert abs(saved["part1-build"]["hit"] - part1["hit"]) < 1e-12
assert abs(saved["part1-build"]["coverage"] - part1["coverage"]) < 1e-12
assert abs(saved["best-single-path"]["hit"] - best["hit"]) < 1e-12
print(f"Part-1 build, recomputed live: hit@10 {part1['hit']:.3f}, coverage {part1['coverage']:.4f} "
      "(== readouts.json)\n")

print(f"{'system':<18} {'hit@10':>7} {'coverage':>9} {'head-share':>11} {'item Gini':>10} "
      f"{'novelty':>8} {'diversity':>10}")
for name in ("full-system", "best-single-path", "part1-build"):
    m = saved[name]
    print(f"{name:<18} {m['hit']:>7.3f} {m['coverage']:>9.4f} {m['head_share']:>11.3f} "
          f"{m['exposure_gini']:>10.3f} {m['novelty']:>8.2f} {m['diversity']:>10.3f}")

Part-1 build, recomputed live: hit@10 0.306, coverage 0.3330 (== readouts.json)

system              hit@10  coverage  head-share  item Gini  novelty  diversity
full-system          0.356    0.2025       0.847      0.913     8.57      0.857
best-single-path     0.362    0.1920       0.862      0.922     8.48      0.850
part1-build          0.306    0.3330       0.793      0.908     8.49      0.871


Against the Part-1 build the full system gains **+0.050 hit** (0.356 vs 0.306) -- and pays for it in
reach: coverage falls from **0.333 to about 0.20**, head-share rises from 0.79 to 0.85, and the item Gini
is a little higher. Novelty (how unpopular the shown books are) and diversity move only slightly; the full
system is *slightly* more novel than the Part-1 build, but far narrower. The best single path (the hybrid)
sits in between. Part 2's models bought accuracy, mostly by concentrating on books readers already read.

## 9. The frontier

No single row wins every column, so stop asking "which is best?" and ask **which systems are not beaten
on both hit and coverage at once.** A system is **dominated** if another one has at least its hit *and*
at least its coverage (and is better on one). The systems left over form the **frontier**: every one of
them is a defensible choice, and moving along it trades one goal for the other.

Two more systems belong on the map, both cheap now that the paths are loaded:

- **dense-heavy weights:** the same 8-path blend, but weighting the dense paths up (two-tower and hybrid
  1.5) and the content and popularity paths down (popularity 0.25, lexical and semantic 0.5);
- **a content-only reranker:** Unit 11's best ranker -- the same reranker over the same 8-path pool, but
  trained on the **content** features only (genre, author and popularity fit to the reader's history).
  It is a sub-second refit of the MLP on the **cached training matrix** -- no path is refit.

In [14]:
dense_weights = {name: 1.0 for name in system.full8}
dense_weights.update({"popularity": 0.25, "lexical": 0.5, "semantic": 0.5, "two-tower": 1.5, "hybrid": 1.5})
dense_heavy = system.evaluate(FixedBlendPath(system.full8, dense_weights, pool=POOL))

X, y = cache.training_matrix("reranker-full8")
content_model = RerankerModel(seed=SEED).fit(X, y, feature_mask=family_mask(("content",), path_names))
content_reranker = NeuralRerankerPath(
    system.full8, lambda subset: system.full8, catalog, pool=POOL, seed=SEED, path_names=path_names,
).load(cache.artifact("reranker-full8")).set_model(content_model)   # the factory is never called
content_only = system.evaluate(content_reranker)

systems = {
    "Part-1 build (U6)": part1,
    "U10 hybrid": best,
    "fixed 8-path blend": fixed,
    "full stack (reranker)": full,
    "dense-heavy weights": dense_heavy,
    "content-only reranker": content_only,
}


def dominated(a, b):
    # True if system b is at least as good as a on hit AND coverage, and better on one.
    at_least = b["hit"] >= a["hit"] and b["coverage"] >= a["coverage"]
    return at_least and (b["hit"] > a["hit"] or b["coverage"] > a["coverage"])


print(f"{'system':<24} {'hit@10':>7} {'coverage':>9} {'head-share':>11} {'item Gini':>10}  frontier?")
for name, m in systems.items():
    beaten_by = [other for other, n in systems.items() if dominated(m, n)]
    verdict = "yes" if not beaten_by else f"no (beaten by {beaten_by[0]})"
    print(f"{name:<24} {m['hit']:>7.3f} {m['coverage']:>9.4f} {m['head_share']:>11.3f} "
          f"{m['exposure_gini']:>10.3f}  {verdict}")

system                    hit@10  coverage  head-share  item Gini  frontier?
Part-1 build (U6)          0.306    0.3330       0.793      0.908  yes
U10 hybrid                 0.362    0.1920       0.862      0.922  no (beaten by fixed 8-path blend)
fixed 8-path blend         0.362    0.2060       0.846      0.923  yes
full stack (reranker)      0.356    0.2025       0.847      0.913  no (beaten by fixed 8-path blend)
dense-heavy weights        0.376    0.1835       0.885      0.926  yes
content-only reranker      0.384    0.1365       0.960      0.940  yes


Read the frontier from the widest system to the most accurate:

- **Part-1 build (0.306 / 0.333):** the *oldest* system has by far the widest reach. Nothing in Part 2
  matches its coverage.
- **Fixed 8-path blend (0.362 / 0.206):** more hit, much less reach. It dominates both the hybrid (same
  hit, less coverage) and the full stack with its reranker (less of both).
- **Dense-heavy weights (0.376 / about 0.184):** more hit again, by leaning on the dense paths -- and less
  reach, with head-share up to about 0.89.
- **Content-only reranker (0.384 / about 0.137):** the best hit in the whole book -- and its most
  concentrated system, with about **96%** of all slots on the 200 head books.

So the honest summary of fourteen units is: **no system dominates.** More machinery did not
automatically win -- the full stack with its learned reranker is *off* the frontier -- and every gain in
hit after the Part-1 build was paid for in reach and exposure. The one learned stage that *does* add
hit, the content-only reranker, does it by matching readers to the genres and authors they already read,
which is also the most concentrating thing a recommender can do.

Which point on the frontier should a library ship? That is not a question the code can answer. A
library that wants readers to find backlist and new authors would pick near the Part-1 end; a shop that
measures success by "they read what we suggested" would pick near the content-reranker end. The code's
job is to make every point's cost visible.

## Where this goes

- **The system:** 8 retrieval paths (the two-tower served through HNSW, the warm feature tower for cold
  books) -> a 50-per-path pool -> a calibrated fixed blend (0.362 / 0.206) or the learned reranker
  (0.356 / 0.203). Expensive models come from a seeded, checked cache; MF and the two-tower refit on
  screen to the same arrays.
- **What each stage bought:** HNSW -- speed, no accuracy (recall about 0.999). The feature tower --
  113 cold books served, the only learned path to reach them; hard negatives trade warm hit (0.298 to
  0.172) for cold coverage (0.138 to 0.226). The learned reranker -- nothing on this data (-0.006).
  Each single path -- within about 0.01 (fixed-blend leave-one-out; popularity's removal even helps).
- **The sequence path** scores about 0.423 on the session log -- a different population, a different
  surface, its own table.
- **The benchmark:** +0.050 hit over the Part-1 build, at a coverage cost of about 0.13.
- **The frontier:** Part-1 build, fixed blend, dense-heavy weights, content-only reranker -- from
  widest to most accurate. Nothing dominates.

### The bridge to the capstone

The capstone, **`project-14-bookrec`**, hands you the same cached models and asks you to build *your*
recommender, and defend it: pick and justify your retrieval paths, build the pool and blend, decide
whether a reranker earns its place, produce your own ablation table and beyond-accuracy table, benchmark
against the Part-1 build, and write a design memo on the trade-offs -- accuracy vs reach vs exposure,
cold start, and cost. There is no single right answer. There is a frontier, and a choice on it that you
can defend with numbers.